# Multimodal Maestro: Using Early Stopping for Efficient Training

This notebook demonstrates how to use the early stopping feature with Multimodal Maestro models to reduce training time and prevent overfitting.

## Introduction

Early stopping is a regularization technique to prevent overfitting in machine learning models. It works by monitoring a validation metric (typically validation loss) and stopping training when the model performance on the validation set stops improving for a specified number of epochs.

Benefits of early stopping:
1. Reduces training time
2. Prevents overfitting
3. Automatically determines optimal training duration

In this notebook, we'll demonstrate how to enable early stopping with Florence-2 model training.

In [ ]:
# Install required packages
%pip install multimodal-maestro supervision --quiet

In [ ]:
# Import necessary libraries
import os

from maestro.trainer.common.metrics import MeanAveragePrecisionMetric
from maestro.trainer.models.florence_2.core import Florence2Configuration, train

## Downloading a sample dataset

For this example, we'll use a small object detection dataset. You can replace this with your own dataset.

In [ ]:
# Download a sample dataset (chess pieces dataset)
%pip install roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")  # Replace with your API key or remove if using public datasets
project = rf.workspace("roboflow-100").project("chess-pieces-detection")
dataset = project.version(2).download("coco")

## Configuring Training with Early Stopping

Now we'll set up the training configuration with early stopping enabled.

In [ ]:
# Get the dataset path
dataset_path = os.path.join(os.getcwd(), dataset.location)

# Configure the training
config = Florence2Configuration(
    dataset=dataset_path,
    epochs=20,  # Set a large enough number of epochs
    batch_size=2,  # Use a small batch size for this example
    lr=1e-5,
    optimization_strategy="lora",
    metrics=[MeanAveragePrecisionMetric()],
    # Early stopping configuration
    early_stopping=True,  # Enable early stopping
    early_stopping_patience=3,  # Stop after 3 epochs with no improvement
    early_stopping_threshold=0.01,  # Minimum change to be considered as improvement
    early_stopping_monitor="val_loss",  # Metric to monitor
)

## Training the Model with Early Stopping

Now we'll start training the model. With early stopping enabled, training will automatically stop once the validation loss stops improving for 3 consecutive epochs.

In [ ]:
# Train the model
train(config)

## Visualizing Training Metrics

After training completes, you can examine the training and validation metrics to see how early stopping worked.

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd

# Find the most recent training run
runs = sorted(glob.glob("./training/florence_2/*"))
latest_run = runs[-1] if runs else None

if latest_run:
    # Try to load the metrics
    try:
        metrics_dir = os.path.join(latest_run, "metrics")
        train_loss = pd.read_csv(os.path.join(metrics_dir, "train_loss.csv"))
        val_loss = pd.read_csv(os.path.join(metrics_dir, "val_loss.csv"))

        # Plot training and validation loss
        plt.figure(figsize=(10, 5))
        plt.plot(train_loss["epoch"], train_loss["value"], label="Training Loss")
        plt.plot(val_loss["epoch"], val_loss["value"], label="Validation Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.title("Training and Validation Loss (with Early Stopping)")
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.show()

        # Show where early stopping occurred
        best_epoch = val_loss["value"].idxmin()
        print(f"Best epoch: {best_epoch}")
        print(f"Best validation loss: {val_loss['value'].min()}")
        print(f"Training stopped at epoch: {val_loss['epoch'].max()}")
    except Exception as e:
        print(f"Could not load metrics: {e}")
else:
    print("No training runs found")

## Conclusion

In this notebook, we've demonstrated how to use early stopping with the Florence-2 model in Multimodal Maestro. The same approach can be applied to other models like PaliGemma-2, and Qwen2.5-VL.

Early stopping is a valuable technique for efficient model training, as it:

1. Saves training time and computational resources
2. Automatically determines the optimal number of training epochs
3. Helps prevent overfitting

By adjusting the `early_stopping_patience`, `early_stopping_threshold`, and `early_stopping_monitor` parameters, you can fine-tune the early stopping behavior to suit your specific training needs.